In [7]:
import numpy as np
from scipy.sparse import load_npz

X_train = load_npz('../data/features/X_satisfaction_train.npz')
X_val   = load_npz('../data/features/X_satisfaction_val.npz')
X_test  = load_npz('../data/features/X_satisfaction_test.npz')

y_train = np.load('../data/features/y_satisfaction_train.npy') - 1
y_val   = np.load('../data/features/y_satisfaction_val.npy') - 1
y_test  = np.load('../data/features/y_satisfaction_test.npy') - 1

print(X_train.shape, y_train.shape)

(140080, 125) (140080,)


In [8]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score

xgb_model = XGBClassifier(
    objective='multi:softmax',
    num_class=5,
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)

y_val_pred = xgb_model.predict(X_val)
print("Validation Accuracy:", accuracy_score(y_val, y_val_pred))
print(classification_report(y_val, y_val_pred))

Validation Accuracy: 0.19776069518716577
              precision    recall  f1-score   support

           0       0.19      0.19      0.19      5887
           1       0.20      0.20      0.20      5994
           2       0.20      0.21      0.21      6022
           3       0.21      0.19      0.20      6100
           4       0.19      0.20      0.20      5917

    accuracy                           0.20     29920
   macro avg       0.20      0.20      0.20     29920
weighted avg       0.20      0.20      0.20     29920



In [9]:
import joblib
joblib.dump(xgb_model, '../saved_models/satisfaction_classifier_baseline.pkl')

['../saved_models/satisfaction_classifier_baseline.pkl']

In [10]:
import pandas as pd
import joblib
import torch
from sklearn.model_selection import train_test_split

df = pd.read_csv('../data/features/featured_data.csv')

text_data       = df['issue_description']
structured_data = df.drop(columns=['issue_description', 'priority',
                                    'sla_breached', 'customer_satisfaction_score'])

print(text_data.shape, structured_data.shape)

(200000,) (200000, 20)


In [11]:
vocab = joblib.load('../saved_models/priority_vocab.pkl')

def text_to_sequence(text, vocab):
    return [vocab.get(word, 0) for word in str(text).split()]

MAX_LEN   = 13
sequences = [text_to_sequence(text, vocab) for text in text_data]
padded    = np.array([seq + [0] * (MAX_LEN - len(seq)) for seq in sequences])

y_train = np.load('../data/features/y_satisfaction_train.npy')
y_val   = np.load('../data/features/y_satisfaction_val.npy')
y_test  = np.load('../data/features/y_satisfaction_test.npy')

X_text_tv, X_text_test, X_struct_tv, X_struct_test = train_test_split(
    padded, structured_data, test_size=0.15, random_state=42
)
X_text_train, X_text_val, X_struct_train, X_struct_val = train_test_split(
    X_text_tv, X_struct_tv, test_size=0.176, random_state=42
)

print(X_text_train.shape, X_struct_train.shape, y_train.shape)

(140080, 13) (140080, 20) (140080,)


In [12]:
ohe_satisfaction    = joblib.load('../saved_models/ohe_sla_satisfaction.pkl')
scaler_satisfaction = joblib.load('../saved_models/scaler_satisfaction.pkl')

cat_cols = [c for c in X_struct_train.select_dtypes('str').columns]

def encode_cats(X, ohe, cat_cols):
    encoded    = ohe.transform(X[cat_cols])
    encoded_df = pd.DataFrame(encoded, columns=ohe.get_feature_names_out(cat_cols), index=X.index)
    return pd.concat([X.drop(columns=cat_cols), encoded_df], axis=1)

X_struct_train = encode_cats(X_struct_train, ohe_satisfaction, cat_cols)
X_struct_val   = encode_cats(X_struct_val,   ohe_satisfaction, cat_cols)
X_struct_test  = encode_cats(X_struct_test,  ohe_satisfaction, cat_cols)

num_cols = ['customer_age', 'customer_tenure_months', 'previous_tickets',
            'first_response_time_hours', 'day_of_week', 'month', 'ticket_rate']

X_struct_train[num_cols] = scaler_satisfaction.transform(X_struct_train[num_cols])
X_struct_val[num_cols]   = scaler_satisfaction.transform(X_struct_val[num_cols])
X_struct_test[num_cols]  = scaler_satisfaction.transform(X_struct_test[num_cols])

print(X_struct_train.shape)

(140080, 75)


In [16]:
X_text_train_tensor  = torch.LongTensor(X_text_train)
X_text_val_tensor    = torch.LongTensor(X_text_val)
X_text_test_tensor   = torch.LongTensor(X_text_test)

X_struct_train_tensor = torch.FloatTensor(X_struct_train.values)
X_struct_val_tensor   = torch.FloatTensor(X_struct_val.values)
X_struct_test_tensor  = torch.FloatTensor(X_struct_test.values)

y_train_tensor = torch.LongTensor(y_train - 1)
y_val_tensor   = torch.LongTensor(y_val - 1)
y_test_tensor  = torch.LongTensor(y_test - 1)

print(X_text_train_tensor.shape, X_struct_train_tensor.shape)

torch.Size([140080, 13]) torch.Size([140080, 75])


In [14]:
import torch.nn as nn

class DualInputModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, struct_input_dim, num_classes):
        super(DualInputModel, self).__init__()

        self.embedding = nn.Embedding(vocab_size + 1, embed_dim, padding_idx=0)
        self.lstm      = nn.LSTM(embed_dim, hidden_dim, batch_first=True)

        self.struct_fc = nn.Sequential(
            nn.Linear(struct_input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim + 64, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, text, struct):
        embedded       = self.embedding(text)
        _, (hidden, _) = self.lstm(embedded)
        text_out       = hidden[-1]
        struct_out     = self.struct_fc(struct)
        combined       = torch.cat([text_out, struct_out], dim=1)
        return self.classifier(combined)

In [17]:
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import classification_report, accuracy_score

model = DualInputModel(
    vocab_size=len(vocab),
    embed_dim=32,
    hidden_dim=64,
    struct_input_dim=75,
    num_classes=5
)

train_loader = DataLoader(TensorDataset(X_text_train_tensor, X_struct_train_tensor, y_train_tensor),
                          batch_size=64, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_text_val_tensor,   X_struct_val_tensor,   y_val_tensor),
                          batch_size=64, shuffle=False)
test_loader  = DataLoader(TensorDataset(X_text_test_tensor,  X_struct_test_tensor,  y_test_tensor),
                          batch_size=64, shuffle=False)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = model.to(device)

for epoch in range(5):
    model.train()
    train_loss, correct, total = 0, 0, 0

    for text_b, struct_b, y_b in train_loader:
        text_b, struct_b, y_b = text_b.to(device), struct_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        out  = model(text_b, struct_b)
        loss = criterion(out, y_b)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        _, pred = out.max(1)
        correct += pred.eq(y_b).sum().item()
        total   += y_b.size(0)

    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for text_b, struct_b, y_b in val_loader:
            text_b, struct_b, y_b = text_b.to(device), struct_b.to(device), y_b.to(device)
            _, pred = model(text_b, struct_b).max(1)
            val_correct += pred.eq(y_b).sum().item()
            val_total   += y_b.size(0)

    print(f"Epoch {epoch+1}/5 | Loss: {train_loss/len(train_loader):.4f} | "
          f"Train Acc: {correct/total:.4f} | Val Acc: {val_correct/val_total:.4f}")

# Test evaluation
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for text_b, struct_b, y_b in test_loader:
        text_b, struct_b, y_b = text_b.to(device), struct_b.to(device), y_b.to(device)
        _, pred = model(text_b, struct_b).max(1)
        all_preds.extend(pred.cpu().numpy())
        all_labels.extend(y_b.cpu().numpy())

print(f"Test Accuracy: {accuracy_score(all_labels, all_preds):.4f}")
print(classification_report(all_labels, all_preds))

# Save
torch.save(model.state_dict(), '../saved_models/satisfaction_classifier_dl.pth')

Epoch 1/5 | Loss: 1.6102 | Train Acc: 0.1993 | Val Acc: 0.2043
Epoch 2/5 | Loss: 1.6096 | Train Acc: 0.2018 | Val Acc: 0.1945
Epoch 3/5 | Loss: 1.6095 | Train Acc: 0.2015 | Val Acc: 0.1994
Epoch 4/5 | Loss: 1.6092 | Train Acc: 0.2055 | Val Acc: 0.1979
Epoch 5/5 | Loss: 1.6090 | Train Acc: 0.2080 | Val Acc: 0.1977
Test Accuracy: 0.1949
              precision    recall  f1-score   support

           0       0.20      0.04      0.07      6059
           1       0.19      0.31      0.24      5933
           2       0.19      0.30      0.24      5989
           3       0.19      0.03      0.06      6035
           4       0.20      0.29      0.23      5984

    accuracy                           0.19     30000
   macro avg       0.20      0.20      0.17     30000
weighted avg       0.20      0.19      0.17     30000

